## Preparação e Tratamento de Dados

**Objetivo:** transformar os arquivos brutos (`funcionarios`, `departamentos`, `filiais`) em uma única tabela analítica, `rh_dados_limpos`, com **uma linha por colaborador**, pronta para os testes de hipótese do notebook `02_analise.ipynb`.

**Fluxo:** `data/raw/*.csv` → tabelas no DuckDB (raw) → diagnóstico → `sql/limpeza.sql` → validação → `data/processed/rh_dados_limpos.{csv,parquet}`

### Decisão técnica: SQL com DuckDB

Para a etapa de tratamento, padronização e criação de variáveis analíticas, optei por utilizar a linguagem **SQL via DuckDB**. Essa decisão técnica foi pautada por três motivos principais:

* **Leitura direta de arquivos:** `read_csv_auto` carrega os CSVs sem etapas intermediárias.
* **Auditabilidade/Transparência:** a lógica de transformação fica em `sql/limpeza.sql`, versionado com o projeto, separada da narrativa do notebook.
* **Escalabilidade:** o DuckDB é orientado a colunas (OLAP). Com o volume atual o ganho de desempenho é irrelevante, mas o mesmo pipeline funcionaria com bases muito maiores.


### Preparação do ambiente - Carga de Dados Brutos

Os três arquivos `.csv` são carregados no DuckDB (`data/rh_dados.duckdb`) **sem nenhuma alteração**, para que todo o tratamento posterior seja feito em SQL e a origem fique preservada.

**Objetivo:** Criar a camada de staging para permitir todo o processo de preparação seja executado a partir da linguagem **SQL**.

In [5]:
import pandas as pd
import duckdb

con = duckdb.connect('../data/rh_dados.duckdb')

con.execute("""
    CREATE OR REPLACE TABLE funcionarios AS 
    SELECT * FROM read_csv_auto('../data/raw/funcionarios.csv');
""")

con.execute("""
    CREATE OR REPLACE TABLE departamentos AS 
    SELECT * FROM read_csv_auto('../data/raw/departamentos.csv');
""")

con.execute("""
    CREATE OR REPLACE TABLE filiais AS 
    SELECT * FROM read_csv_auto('../data/raw/filiais.csv');
""")


### Visualização para Exploração de Dados

Antes de limpar, as inconsistências foram mapeadas com queries **SQL**. O que foi encontrado:

| Problema | Coluna(s) | 
|---|---|
| Rótulo inconsistente (`Fem`) | `genero` |
| Datas no formato DD/MM/AAAA | `data_admissao`, `data_promocao` | 
| Salário como texto com prefixo `R$ ` e vírgula decimal | `salario_base` | 
| Coluna `nome`, irrelevante para a análise | `nome` | 

A fim de garantir a integridade da base, os dados devem ser verificados contra registros duplicados também. 

In [6]:
df_sujo = con.execute("""
SELECT * FROM funcionarios
""").df()

pd.set_option('display.max_rows', 10)
df_sujo

,id_colaborador,nome,genero,nivel,data_admissao,data_promocao,salario_base,percentual_bonus,id_departamento,id_filial,id_reporta_a,processos_actifs,horas_extras_mes,score_satisfacao,home_office,status_atual
0,1,Isabela Carvalho 1,Não-binário,Júnior,07/09/2024,NaN,"R$ 3834,82",0,10,1,501,14,8,NaN,False,Ativo
1,2,Samuel Teixeira 2,Não-binário,Pleno,2022-05-09,NaN,7267.8,15,10,2,300,16,8,4.9,True,Desligado
2,3,Diego Santos 3,Masculino,Júnior,2025-01-02,NaN,4771.24,1,10,1,501,29,11,4.7,True,Ativo
3,4,Carla Oliveira 4,Não-binário,Pleno,2020-04-03,NaN,7819.21,14,30,2,300,32,13,3.5,False,Desligado
4,5,Carla Oliveira 5,Não-binário,Júnior,2025-09-15,NaN,4452.47,5,20,3,300,46,42,2.2,False,Desligado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
645,646,Bruno Souza 646,Masculino,Júnior,2024-08-05,NaN,3519.69,9,10,1,300,28,18,4.7,True,Ativo
646,647,Diego Santos 647,Não-binário,Pleno,2020-09-23,NaN,6590.93,16,40,2,502,14,10,NaN,True,Desligado
647,648,Olivia Barbosa 648,Feminino,Sócio,2013-03-14,2017-03-04,18228.69,51,30,1,<NA>,15,19,3.4,True,Ativo
648,649,Mariana Rocha 649,Feminino,Pleno,2022-10-25,NaN,6501.22,16,40,2,501,21,8,3.9,True,Desligado


### Limpeza, Engenharia de Variáveis e Criação da Tabela Analítica (`sql/limpeza.sql`)

A preparação dos dados foi estruturada com os seguintes focos:

* **Anonimização:** Remoção de dados sensíveis ou irrelevantes para a agregação (ex: nomes).
* **Padronização:** Padronização de textos, tratamento de tipos e de formatação (Ex: datas) e limpeza de strings numéricas (ex: retirada de prefixo de moeda).
* **Engenharia de Variáveis (Feature Engineering):** Criação de colunas binárias (flags) para facilitar a aplicação de testes estatísticos posteriores (ex: `desligado`, `promocao`, `bonus`) e cálculo da remuneração total.

Para manter o notebook visualmente limpo, a query SQL responsável por criar a tabela final unificada e tratada foi isolada no arquivo `sql/limpeza.sql`. No padrão analítico que estamos adotando, a query irá consultar os DataFrames e gerar um único DataFrame final limpo, `rh_dados_limpos`.

In [7]:
with open('../sql/limpeza.sql', 'r', encoding='utf-8') as f:
    query_limpeza = f.read()

con.execute(query_limpeza)

df = con.execute("SELECT * FROM rh_dados_limpos").df()
pd.set_option('display.max_columns', None)
df

,id_colaborador,genero,nivel,data_admissao,data_promocao,promocao,salario_base,percentual_bonus,remuneracao_total,id_departamento,id_filial,id_reporta_a,processos_ativos,horas_extras_mes,score_satisfacao,home_office,status_atual,desligado,bonus,nome_departamento,id_chefe_departamento,cidade,estado,id_socio_diretor
0,1,Não-binário,Júnior,2024-09-07,NaT,0,3834.82,0.0,3834.82,10,1,501,14,8.0,NaN,0,Ativo,0,0,Civil,100,Salvador,BA,501
1,2,Não-binário,Pleno,2022-05-09,NaT,0,7267.80,15.0,8357.97,10,2,300,16,8.0,4.9,1,Desligado,1,1,Civil,100,São Paulo,SP,502
2,3,Masculino,Júnior,2025-01-02,NaT,0,4771.24,1.0,4818.95,10,1,501,29,11.0,4.7,1,Ativo,0,1,Civil,100,Salvador,BA,501
3,4,Não-binário,Pleno,2020-04-03,NaT,0,7819.21,14.0,8913.90,30,2,300,32,13.0,3.5,0,Desligado,1,1,Corporativo,300,São Paulo,SP,502
4,5,Não-binário,Júnior,2025-09-15,NaT,0,4452.47,5.0,4675.09,20,3,300,46,42.0,2.2,0,Desligado,1,1,Trabalhista,200,Rio de Janeiro,RJ,503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
645,646,Masculino,Júnior,2024-08-05,NaT,0,3519.69,9.0,3836.46,10,1,300,28,18.0,4.7,1,Ativo,0,1,Civil,100,Salvador,BA,501
646,647,Não-binário,Pleno,2020-09-23,NaT,0,6590.93,16.0,7645.48,40,2,502,14,10.0,NaN,1,Desligado,1,1,Tributário,400,São Paulo,SP,502
647,648,Feminino,Sócio,2013-03-14,2017-03-04,1,18228.69,51.0,27525.32,30,1,<NA>,15,19.0,3.4,1,Ativo,0,1,Corporativo,300,Salvador,BA,501
648,649,Feminino,Pleno,2022-10-25,NaT,0,6501.22,16.0,7541.42,40,2,501,21,8.0,3.9,1,Desligado,1,1,Tributário,400,São Paulo,SP,502


### Exportação

Exportação dos dados limpos em formado `.csv` e  `parquet`.

Os dados foram exportados em dois formatos distintos, visando diferentes cenários de consumo:

* **Formato CSV:** (`;`, `utf-8-sig`) abre corretamente no Excel em português e é fácil de inspecionar.
* **Formato Parquet:** formato colunar compactado que preserva os tipos (datas, inteiros, decimais). O uso do Parquet é o padrão na indústria de dados e garante uma ingestão muito mais eficiente e segura na ferramenta de visualização (PowerBI e Streamlit).

In [8]:
df.to_csv('../data/processed/rh_dados_limpos.csv', index=False, sep=';', decimal='.', encoding='utf-8-sig')

df.to_parquet('../data/processed/rh_dados_limpos.parquet', index=False, engine='pyarrow')

print(f"Total de registros exportados: {len(df)}")
con.close()

Total de registros exportados: 650
